<a href="https://colab.research.google.com/github/salomesanchez160/Analitica-de-Negocios/blob/main/%C3%81rbol_Heart_Diseasepynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Caso de Estudio**

Una institución de salud quiere implementar un modelo de árbol de decisión
para predecir si un paciente va a desarrollar una enfermedad cardiaca, con base
en sus indicadores clínicos. Variables utilizadas:

* Age (Edad): Edad del paciente en años.
* Systolic (Presión Sistólica): Presión arterial sistólica en mm Hg.
* Diastolic (Presión Diastólica): Presión arterial diastólica en mm Hg.
* BMI (Índice de Masa Corporal): Índice de masa corporal del paciente.
* Disease: Variable objetivo — 0: Sin enfermedad, 1: Con enfermedad.

0. Se procede con la carga de librerías de trabajo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

1. Se procede con la carga de los datos de trabajo

In [ ]:
nxl= ('/content/2. HeartDisease.xlsx')
XDB= pd.read_excel(nxl, sheet_name=0)

#Selección de variables predictorias y objetivo
XD = XDB[['Age', 'Systolic', 'Diastolic', 'BMI']]
yd = XDB['Disease']

display (XD)

,Age,Systolic,Diastolic,BMI
0,44,112,111,17
1,55,128,90,27
2,47,131,94,26
3,31,151,104,17
4,65,148,117,17
...,...,...,...,...
175,43,119,76,25
176,63,107,113,40
177,47,116,61,27
178,70,110,88,24


2. Se procede con la implementación del modelo del árbol

In [ ]:
mar= DecisionTreeClassifier(criterion= 'gini', max_depth= 4 )
mar.fit (XD, yd) #Aquí el modelo busca relación entrada-salida

#¿Y que fue lo que hizo el modelo?
ydp= mar.predict (XD) #Esto es lo que pronóstica el modelo

#Se construye la matriz de confusión
cm= confusion_matrix (yd, ydp)
display (cm)
VN=cm [0,0]; FP=cm [0,1]; FN=cm[1,0]; VP=cm[1,1]

#Métricas de desempeño
Ex= (VP+VN)/len(XD) #1. Exactitud: Comportamiento general
print ("La exactitud es: ", Ex)
Sen= VP/(VP+FN) #2. Sensibilidad: Como detecta personas enfermas
print ("La sensibilidad es: ", Sen)
Spe= VN/(VN+FP) #3. Especificidad: Como detecta personas sanas
print ("La especificidad es: ", Spe)
Pre= VP/(VP+FP) #4. Precisión: ¿Qué tan preciso es el pronóstico?
print ("La precisión es: ", Pre)
PreNeg= VN/(VN+FN) #5. Precisión negativa

array([[63, 10],
       [20, 87]])

La exactitud es:  0.8333333333333334
La sensibilidad es:  0.8130841121495327
La especificidad es:  0.863013698630137
La precisión es:  0.8969072164948454


3. Despliegue del árbol de decisiones

In [ ]:
from sklearn.tree import export_graphviz #Exporta los datos de un gráfico
from pydotplus import graph_from_dot_data #Es un graficador

vs=["Age", "Systolic", "Diastolic", "BMI" ] #Títulos del árbol
dot_data= export_graphviz (mar, feature_names= vs) #Exportar de números a gráfico en pdf
graph= graph_from_dot_data (dot_data) #Crear el gráfico
graph.write_png('Árbol_Heart_Disease.png')

True

4. Pronóstico para nuevos pacientes (Hoja 2: Datos Pronóstico)

In [ ]:
XDB2 = pd.read_excel(nxl, sheet_name=1)
X_nuevo = XDB2[['Age', 'Systolic', 'Diastolic', 'BMI']]

pronostico= mar.predict(X_nuevo)
probabilidad= mar.predict_proba(X_nuevo)

XDB2['Pronóstico']= pronostico
XDB2['Prob. Sin Enf.']= probabilidad[:, 0].round(4)
XDB2['Prob. Con Enf.']= probabilidad[:, 1].round(4)
XDB2['Clasificación']= XDB2['Pronóstico'].map({0: 'Sin Enfermedad', 1: 'Con Enfermedad'})

display(XDB2)

,Unnamed: 0,Age,Systolic,Diastolic,BMI,Pronóstico,Prob. Sin Enf.,Prob. Con Enf.,Clasificación
0,1,33,120,64,33,0,0.7576,0.2424,Sin Enfermedad
1,2,55,115,81,32,0,0.6000,0.4000,Sin Enfermedad
2,3,51,130,70,20,0,0.7576,0.2424,Sin Enfermedad
3,4,65,101,63,18,1,0.0625,0.9375,Con Enfermedad
4,5,62,109,110,38,1,0.0000,1.0000,Con Enfermedad
5,6,60,130,69,35,1,0.0625,0.9375,Con Enfermedad
6,7,61,124,95,36,1,0.0000,1.0000,Con Enfermedad
7,8,53,150,81,41,1,0.0000,1.0000,Con Enfermedad


###**Análisis de Resultados**

De la base de datos se observan 180 pacientes, de los cuales 73 no presentan
enfermedad cardiaca (0) y 107 sí la presentan (1).

De la matriz de confusión se obtiene:
* VN = 63: Pacientes sanos correctamente identificados.
* FP = 10: Pacientes sanos clasificados erróneamente como enfermos.
* FN = 20: Pacientes enfermos no detectados por el modelo.
* VP = 87: Pacientes enfermos correctamente identificados.

Con respecto a las métricas, el modelo logró una exactitud del 83.3%, lo que indica un buen comportamiento general, superando el umbral mínimo del 75% definido para modelos de clasificación. La especificidad (86.3%) y la precisión positiva (89.7%) destacan como las métricas más altas, indicando que el modelo es muy confiable cuando clasifica a un paciente como enfermo.

La variable raíz del árbol es la Edad (Age ≤ 54.50), siendo el predictor con mayor poder discriminatorio. El lado derecho del árbol (Age > 54.50) revela:

* Nodo puro 1 (Gini=0.0000): SI Age > 54.50 AND Diastolic > 85.50

N=35 pacientes, TODOS con enfermedad cardiaca (100%).
Regla dominante: edad avanzada + presión diastólica alta = enfermedad segura.

* Nodo Puro 2 (Gini=0.0000): SI Age > 54.50 AND Diastolic ≤ 85.50 AND Diastolic > 78.50

N=35 pacientes, todos con enfermedad (100%).

* Hoja extrema (93.8%): SI Age > 54.50 AND Diastolic ≤ 78.50 (hoja nivel 4)

15 de 16 pacientes con enfermedad. Alta presión diastólica Y baja: ambos extremos en pacientes mayores de 54 años indican alto riesgo.

* Nodo puro sin enfermedad: SI Age > 69.50 → N=2, sin enfermedad (100%).

La presión diastólica es la segunda variable más importante. La combinación
Age > 54.50 con distintos rangos de Diastolic genera las reglas más determinantes
del modelo para la detección de enfermedad cardiaca.